# Survival Analysis

This notebook integrates longitudinal FEMA-derived temperament deviations with diagnosis timing to perform survival analysis.

Goals:
- Load latent temperament deviations
- Load and prepare survival endpoints (first diagnosis age) for a psychiatric disorder (e.g., ADHD `F90.0`)
- Use Cox proportional hazards model with left truncation and cluster-robust standard errors
- Evaluate PLS trajectory dimensions (`PLS1`, `PLS2`) as predictive risk markers

Inputs:
- `df` with baseline covariates, PCs, genotyping batch, and IDs
- `df_score` containing PLS scores for temperament deviations
- `df_diag` containing diagnosis ages and event indicators


In [ ]:
# ========================================================================
# Imports and setup
# ========================================================================
# Data handling
import pandas as pd
import numpy as np

# ML/statistics
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Survival analysis
from lifelines import CoxPHFitter


In [ ]:
# Linear regression helper (used for model sanity checks)
model = LinearRegression()
scaler = StandardScaler()


Subject data

In [ ]:
# ========================================================================
# Subject data preparation
# ========================================================================
# Load main phenotype data CSV containing demographic covariates, PCs, batch, IDs
df = pd.read_csv("....csv")

# Define columns for genetic principal components and genotyping batch indicators
pc_cols = [col for col in df.columns if 'PC' in col]
batch_cols = [col for col in df.columns if 'genotyping_batch' in col]

# Define visit windows and temperament phenotype names
time_points = ['18_months', '36_months', '60_months']
phenNames = ['emot', 'act', 'shy', 'soc']

# Note: Here we prepare the baseline subject data; this same df is subsequently merged
# with diagnostic and trajectory scores for survival modeling.


Load latent temperament scores

In [ ]:
# Load precomputed PLS scores from previous FEMA + PLS analyses
df_score = pd.read_csv('...csv')


Load diagnostic data

Year of diagnosis

In [ ]:
# Re-load phenotype dataframe from CSV in case of a separate processing path
# (Ensures this block is self-contained and can run from a fresh kernel)
df = pd.read_csv("....csv")

# Extract standard variables as before for reproducibility
pc_cols = [col for col in df.columns if 'PC' in col]
batch_cols = [col for col in df.columns if 'genotyping_batch' in col]

time_points = ['18_months', '36_months', '60_months']
phenNames = ['emot', 'act', 'shy', 'soc']


In [ ]:
# Create minimal covariate frame for survival models
# Includes genomic PCs, genotyping batches, and identifiers

df = df.loc[:, ['ID', 'yob', 'sex', 'fid',
                'PC1', 'PC2', ..., 
                'genotyping_batch_num_1', 'genotyping_batch_num_2',
                ...]]

# After this step df contains only analysis-ready predictors for survival modeling.


In [ ]:
# Specify ICD code of interest for survival event (e.g., F90.0 = ADHD)
disorder = 'F900'

# Load diagnosis table with first-diagnosis age for the selected disorder
# Use tab-separated file if necessary and include only relevant columns
df_diag = pd.read_csv('...csv', delimiter='\t', low_memory=False, usecols=[disorder, 'ID'])


In [ ]:
# ========================================================================
# Prepare survival analysis dataset
# ========================================================================
# Align diagnosis records with score dataset subjects for analysis
# df_score has trajectory-derived features from CCA/FEMA analysis

df_diag_subset = df_diag.copy()
df_diag_subset = df_diag_subset.reindex(np.unique(df_score.index)).fillna(0)

# Keep only individuals with diagnosis event for this disorder
# (stop_age is age at first diagnosis in days)
df_diag_subset_disease = df_diag_subset[df_diag_subset[disorder] > 0]

df_diag_subset_disease.columns = ['stop_age']
# Event indicator: 1 if diagnosed, 0 otherwise
df_diag_subset_disease.loc[:, 'event'] = 1
# Convert days to years as survival time unit
df_diag_subset_disease.loc[:, 'stop_age'] = df_diag_subset_disease.stop_age / 365.25

# Merge with base covariate and genotype data by subject index
df_survival = pd.merge(df, df_diag_subset_disease, left_index=True, right_index=True, how='left')

# Set censored subjects' stop_age to current observation year minus birth year
# assumes last-followup year 2025 for non-diagnosed participants
df_survival.loc[df_survival['stop_age'].isna(), 'stop_age'] = (
    2025 - df_survival.loc[df_survival['stop_age'].isna(), 'yob'])

# Left-truncation start age (entry age) is set to 5 for all participants
# because diagnosis is only considered after age 5 in this study design
df_survival['start_age'] = 5

# Remove invalid records where event/censoring happened before entered age
df_survival = df_survival[df_survival['stop_age'] > df_survival['start_age']]

# Fill missing event values for non-diagnosed participants with 0
df_survival['event'] = df_survival['event'].fillna(0)

# Merge with trajectory deviation scores (PLS1, PLS2, etc.) by subject index
# Keep only subjects with both covariate and score data
df_survival = pd.merge(df_survival, df_score, left_index=True, right_index=True, how='inner')

df_survival.reset_index(drop=False, inplace=True)

# df_survival ready for Cox model fitting with ages, event flag, predictors, and clusters


## Sanity checks before fitting the Cox model
Verify data integrity:
- unique subject IDs
- valid time window (stop_age > start_age)
- binary event flag
- no missing values in key predictors


In [ ]:
# ----------------------------------------------------------------------------
# Data Integrity Checks
# ----------------------------------------------------------------------------
# 1. Ensure each child appears only once in survival data frame (requirement for Cox model)
assert df_survival['ID_2445'].is_unique

# 2. Confirm survival interval is valid for all subjects
assert np.all(df_survival['stop_age'] > df_survival['start_age'])

# 3. Ensure event column is binary
assert set(df_survival['event'].unique()) <= {0, 1}

# 4. Count missing values for critical columns
required_cols = [
    'start_age', 'stop_age', 'event',
    'PLS1', 'PLS2', 'sex', 'yob', 'fid'
]
print(df_survival[required_cols].isna().sum())


## Fit Cox proportional hazards model
Model includes:
- PLS components (`PLS1`, `PLS2`) from trajectory deviation analysis
- sex and genetic ancestry PCs as covariates
- left truncation (`start_age`) and right censoring (`stop_age`)
- strata by birth year and sex to control cohort effects
- cluster-robust SE by family ID (`fid`)


In [ ]:
# Prepare empty results container for Cox coefficients
# Contains hazard ratio, 95% CI and p-value for each predictor
df_all_coeffs = pd.DataFrame(columns=['HR', 'HR_low', 'HR_up', 'P'])


In [ ]:
# Define covariates included in the Cox model
covariates = ['PLS1', 'PLS2', 'sex',
              'PC1', 'PC2', ...]

# Create model input DataFrame with survival columns and eligible predictors
df_model = df_survival[covariates + ['start_age', 'stop_age', 'event', 'yob', 'fid']]

# Ensure in-place modifications do not raise SettingWithCopy warnings
# optional: df_model = df_model.copy()


In [ ]:
# Fit Cox regression with left truncation and cluster-robust standard errors
cph = CoxPHFitter()

cph.fit(
    df_model,
    duration_col='stop_age',
    event_col='event',
    entry_col='start_age',      # Left truncation start age
    strata=['yob', 'sex'],      # Control birth-year cohorts and sex-specific baseline hazards
    cluster_col='fid',          # Robust SE by family cluster
    robust=True
)

# Print model summary for hazard ratios, CI, and p-values
cph.print_summary()
print('Concordance index: ' + str(np.round(cph.concordance_index_, 3)))
